In [2]:
import lightkurve as lk
from astroquery.mast import Observations
from astropy.io import fits


/home/connor_codes/.local/lib/python3.12/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [3]:
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from astropy.io import fits
from astropy.table import vstack
from astroquery.mast import Observations

In [4]:
SECTORS = [1, 2, 3]
TRAIN_DISPOSITIONS = ["CP", "KP", "FP", "FA"]
LABEL_MAP = {"CP": 1, "KP": 1, "FP": 0, "FA": 0}

EXPTIME_RANGE = [119, 121]
TARGET_CHUNK = 400
PRODUCT_CHUNK = 400
MAX_WORKERS = 8

DATA_DIR = Path("data")
FITS_DIR = DATA_DIR / "fits"
CSV_DIR = DATA_DIR / "lightcurves"
TOI_PATH = DATA_DIR / "toi_catalog.csv"
MANIFEST_PATH = DATA_DIR / "training_manifest.csv"

for d in (DATA_DIR, FITS_DIR, CSV_DIR):
    d.mkdir(parents=True, exist_ok=True)

TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
BTJD_OFFSET = 2457000.0

Fetching TOi columns for the TESS dataset

In [5]:
TOI_COLUMNS = [
    "toi", "tid", "tfopwg_disp", "pl_tranmid", "pl_orbper", "pl_trandurh",
    "pl_trandep", "pl_rade", "st_tmag", "st_teff", "st_logg", "st_rad",
]


def load_toi_catalog(path, refresh=False):
    if path.exists() and not refresh:
        return pd.read_csv(path)
    query = f"select {', '.join(TOI_COLUMNS)} from toi"
    resp = requests.get(TAP_URL, params={"query": query, "format": "csv"}, timeout=120)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    df.to_csv(path, index=False)
    return df


toi = load_toi_catalog(TOI_PATH)
toi["label"] = toi["tfopwg_disp"].map(LABEL_MAP)

train_toi = toi[toi["tfopwg_disp"].isin(TRAIN_DISPOSITIONS)].copy()
train_toi["tid"] = train_toi["tid"].astype(np.int64)
train_tics = np.sort(train_toi["tid"].unique())

print(f"TOIs in catalog: {len(toi):,}")
print(f"Resolved TOIs for training: {len(train_toi):,} on {len(train_tics):,} stars")
toi["tfopwg_disp"].value_counts(dropna=False)

TOIs in catalog: 8,151
Resolved TOIs for training: 2,836 on 2,686 stars


tfopwg_disp
PC     4820
FP     1315
CP      815
KP      606
APC     490
FA      100
NaN       5
Name: count, dtype: int64

In [6]:
def query_sector_observations(sector, tic_ids):
    targets = [str(t) for t in tic_ids]
    tables = []
    for start in range(0, len(targets), TARGET_CHUNK):
        obs = Observations.query_criteria(
            obs_collection="TESS",
            dataproduct_type="timeseries",
            provenance_name="SPOC",
            sequence_number=sector,
            t_exptime=EXPTIME_RANGE,
            target_name=targets[start:start + TARGET_CHUNK],
        )
        if len(obs):
            tables.append(obs)
    return vstack(tables) if tables else None


def light_curve_products(obs_table):
    tables = []
    for start in range(0, len(obs_table), PRODUCT_CHUNK):
        products = Observations.get_product_list(obs_table[start:start + PRODUCT_CHUNK])
        lc = Observations.filter_products(
            products, productSubGroupDescription="LC", extension="fits"
        )
        if len(lc):
            tables.append(lc)
    return vstack(tables) if tables else None


sector_products = {}
for sector in SECTORS:
    obs = query_sector_observations(sector, train_tics)
    if obs is None:
        print(f"Sector {sector}: no matching observations")
        continue
    products = light_curve_products(obs)
    sector_products[sector] = products
    print(f"Sector {sector}: {len(obs)} observations, {len(products)} light curve files")

Sector 1: 104 observations, 104 light curve files
Sector 2: 147 observations, 109 light curve files
Sector 3: 138 observations, 94 light curve files


In [7]:
def download_one(uri, local_path):
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path, "cached"
    status, msg, _ = Observations.download_file(uri, local_path=str(local_path), verbose=False)
    if status != "COMPLETE":
        raise RuntimeError(f"{status}: {msg}")
    return local_path, "downloaded"


downloaded = []
failures = []

for sector, products in sector_products.items():
    sector_dir = FITS_DIR / f"s{sector:04d}"
    sector_dir.mkdir(exist_ok=True)
    jobs = {(row["dataURI"], sector_dir / row["productFilename"]) for row in products}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(download_one, uri, path): (uri, path) for uri, path in jobs}
        for future in as_completed(futures):
            uri, path = futures[future]
            try:
                local_path, _ = future.result()
                downloaded.append((sector, local_path))
            except Exception as exc:
                failures.append((sector, uri, str(exc)))

    print(f"Sector {sector}: {sum(s == sector for s, _ in downloaded)} files ready")

if failures:
    print(f"{len(failures)} downloads failed; rerun this cell to retry them")

Sector 1: 104 files ready
Sector 2: 109 files ready
Sector 3: 94 files ready


In [8]:
LC_COLUMNS = [
    "TIME", "SAP_FLUX", "SAP_FLUX_ERR", "SAP_BKG",
    "PDCSAP_FLUX", "PDCSAP_FLUX_ERR", "QUALITY",
    "MOM_CENTR1", "MOM_CENTR2", "POS_CORR1", "POS_CORR2",
]


def fits_to_frame(path):
    with fits.open(path, memmap=False) as hdul:
        header = hdul[0].header
        data = hdul["LIGHTCURVE"].data
        columns = {}
        for name in LC_COLUMNS:
            dtype = np.int32 if name == "QUALITY" else np.float64
            columns[name.lower()] = np.asarray(data[name]).astype(dtype)
    df = pd.DataFrame(columns)
    meta = {
        "tic_id": int(header["TICID"]),
        "sector": int(header["SECTOR"]),
        "camera": int(header["CAMERA"]),
        "ccd": int(header["CCD"]),
    }
    return df, meta


records = []
for sector, fits_path in downloaded:
    out_dir = CSV_DIR / f"s{sector:04d}"
    out_dir.mkdir(exist_ok=True)
    df, meta = fits_to_frame(fits_path)
    csv_path = out_dir / f"TIC_{meta['tic_id']}.csv"
    if not csv_path.exists():
        df.to_csv(csv_path, index=False)
    good = int(((df["quality"] == 0) & np.isfinite(df["pdcsap_flux"])).sum())
    records.append({**meta, "n_cadences": len(df), "n_good": good, "csv_path": str(csv_path)})

lc_index = pd.DataFrame(records).sort_values(["sector", "tic_id"]).reset_index(drop=True)
print(f"{len(lc_index)} light curves converted")
lc_index.head()

307 light curves converted


,tic_id,sector,camera,ccd,n_cadences,n_good,csv_path
0,25155310,1,4,1,20076,18278,data/lightcurves/s0001/TIC_25155310.csv
1,25375553,1,1,3,20076,18233,data/lightcurves/s0001/TIC_25375553.csv
2,29344935,1,1,4,20076,18169,data/lightcurves/s0001/TIC_29344935.csv
3,29781292,1,4,1,20076,18276,data/lightcurves/s0001/TIC_29781292.csv
4,30312676,1,4,1,20076,18256,data/lightcurves/s0001/TIC_30312676.csv
